# 01.01 — Building a Graph Definition

**orthograph** lets you build graph definitions using Pydantic-based classes.
The idea is similar to how [SQLModel](https://sqlmodel.tiangolo.com/) works for SQL tables
or [Pandera](https://pandera.readthedocs.io/) for DataFrames, but for graph data structures:
you declare node types, relationship types, and their constraints as Python classes,
and orthograph validates real data against them.

This notebook walks through the class-based API for **defining** a graph data model.
We use a filmography domain (Person, Movie, City) as a running example.

In [1]:
from typing import Optional

from orthograph.definition import (
    CardinalitySpec,
    GraphDefinition,
    GraphValidationError,
    NodeModel,
    RelationshipModel,
)

## Defining Node Types

`NodeModel` is a Pydantic `BaseModel` subclass. Each concrete node type must set:

- **`__label__`** -- a string that names the node type (analogous to a Neo4j label).
- **`__uid_field__`** *(optional)* -- the name of the property that acts as the unique identifier.

Properties are declared as regular typed Python fields, exactly as you would on any Pydantic model.
Use `Optional[T] = None` for properties that may be absent.

In [2]:
class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"

    name: str
    age: int
    email: Optional[str] = None


class Movie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"

    title: str
    year: int
    rating: Optional[float] = None


class City(NodeModel):
    __label__ = "City"
    __uid_field__ = "name"

    name: str
    country: str

## Defining Relationship Types

`RelationshipModel` is also a Pydantic `BaseModel` subclass. Each relationship type requires:

| Class variable | Purpose |
|---|---|
| `__label__` | Relationship type name (e.g. `"ACTED_IN"`). |
| `__source_label__` | The string label of the source node type (matching `__label__` on the source `NodeModel` subclass). |
| `__target_label__` | The string label of the target node type (matching `__label__` on the target `NodeModel` subclass). |
| `__directed__` | `True` (default) for directed, `False` for undirected. |
| `__source_cardinality__` | How many outgoing relationships of this type each source node may have. |
| `__target_cardinality__` | How many incoming relationships of this type each target node may have. |

Cardinality defaults to `0..*` (`CardinalitySpec(min=0, max=None)`, no constraint). Properties on relationships work identically to node properties.

In [3]:
class ActedIn(RelationshipModel):
    __label__ = "ACTED_IN"
    __source_label__ = "Person"
    __target_label__ = "Movie"

    role: str  # required property


class Directed(RelationshipModel):
    __label__ = "DIRECTED"
    __source_label__ = "Person"
    __target_label__ = "Movie"


class LivesIn(RelationshipModel):
    __label__ = "LIVES_IN"
    __source_label__ = "Person"
    __target_label__ = "City"
    __source_cardinality__ = CardinalitySpec(
        min=1, max=1
    )  # each Person lives in exactly 1 City
    __target_cardinality__ = CardinalitySpec(min=0, max=None)


class FriendOf(RelationshipModel):
    __label__ = "FRIEND_OF"
    __source_label__ = "Person"
    __target_label__ = "Person"
    __directed__ = False  # friendship is symmetric

### Directed vs Undirected Relationships

By default, relationships are **directed** (`__directed__ = True`): `ACTED_IN` goes from
`Person` to `Movie`, and the source/target distinction matters.

Setting `__directed__ = False` declares a **symmetric/undirected** relationship. This
affects several parts of the system:

| Aspect | Directed | Undirected |
|---|---|---|
| **Validation** | Source must match `__source_label__`, target must match `__target_label__` | Either direction is accepted |
| **Cardinality** | Outgoing/incoming counted separately | Both directions combined into a single count |
| **Cypher generation** | Uses `->` arrow | Uses `-` (no arrow) |
| **Mermaid diagrams** | `-->` (arrow) | `---` (no arrow) |
| **Introspection** | Outgoing from source_type, incoming to target_type | Appears in both outgoing and incoming for both endpoint types |

Undirected relationships work with both **same-type** endpoints (like `FRIEND_OF` between
two `Person` nodes) and **cross-type** endpoints (like `COLLABORATES` between `Person` and
`Company`). For cross-type undirected relationships, the validator accepts data stored in
either direction in the database.

## Assembling the GraphDefinition

`GraphDefinition` is the container that registers all node and relationship types.
It validates structural consistency at creation time -- duplicate labels and dangling
node references cause an immediate error.

In [4]:
graph_definition = GraphDefinition(
    name="Filmography",
    node_types=[Person, Movie, City],
    relationship_types=[ActedIn, Directed, LivesIn, FriendOf],
)

print("Node labels:        ", graph_definition.node_labels)
print("Relationship labels: ", graph_definition.relationship_labels)

Node labels:         {'Movie', 'Person', 'City'}
Relationship labels:  {'ACTED_IN', 'DIRECTED', 'LIVES_IN', 'FRIEND_OF'}


## Structural Validation

The model validates itself on creation. You can also call `validate_structure()` explicitly
to get a `ValidationResult` with any warnings (e.g. isolated node types).

If you try to create a model with undefined node references -- for example, a relationship
type pointing to a node type that is not registered -- construction raises a `GraphValidationError`.

In [5]:
# Inspect structural warnings on our valid model
result = graph_definition.validate_structure()
print("Is valid:", result.is_valid)
print("Warnings:", len(result.warnings))
print("Errors:  ", len(result.errors))

Is valid: True
Warnings: 0
Errors:   0


In [6]:
# What happens when a relationship references an unregistered node type?
try:
    bad_model = GraphDefinition(
        name="Broken",
        node_types=[Person],  # Movie is missing!
        relationship_types=[ActedIn],  # ActedIn needs Person -> Movie
    )
except GraphValidationError as exc:
    print("Caught GraphValidationError:")
    for issue in exc.issues:
        print(" ", issue)

Caught GraphValidationError:
  [ERROR] UNDEFINED_NODE_TYPE: Relationship ACTED_IN references undefined target node type: Movie (entity=ACTED_IN)


## Introspection

The model exposes several methods for programmatic introspection:

- `get_node_type(label)` / `get_relationship_type(label)` -- look up a type by its label.
- `get_outgoing_relationship_types(node_type)` -- all relationship types that originate from a given node type.
- `get_property_specs()` / `get_required_property_names()` -- inspect properties on any node or relationship type.
- `get_node_label_enum()` / `get_relationship_label_enum()` -- generate Python `Enum` types from the model's labels.

In [7]:
# Look up a node type by label
person_type = graph_definition.get_node_type("Person")
print("Looked up:", person_type)
print()

# Outgoing relationships from Person
outgoing = graph_definition.get_outgoing_relationship_types(Person)
print("Person outgoing relationships:")
for rt in outgoing:
    print(f"  {rt.__label__}  ->  {rt.__target_label__}")
print()

# Property specs on Person
print("Person property specs:")
for name, info in Person.get_property_specs().items():
    print(f"  {name}: type={info.python_type.__name__}, required={info.is_required}")
print()

# Required properties only
print("Person required properties:", Person.get_required_property_names())
print()

# Enum generation
NodeLabel = graph_definition.get_node_label_enum()
RelLabel = graph_definition.get_relationship_label_enum()
print("NodeLabel enum members: ", list(NodeLabel.__members__.keys()))
print("RelLabel enum members:  ", list(RelLabel.__members__.keys()))

Looked up: <class '__main__.Person'>

Person outgoing relationships:
  ACTED_IN  ->  Movie
  DIRECTED  ->  Movie
  LIVES_IN  ->  City
  FRIEND_OF  ->  Person

Person property specs:
  name: type=str, required=True
  age: type=int, required=True
  email: type=str, required=False

Person required properties: {'age', 'name'}

NodeLabel enum members:  ['Person', 'Movie', 'City']
RelLabel enum members:   ['ACTED_IN', 'DIRECTED', 'LIVES_IN', 'FRIEND_OF']


## Instantiating Nodes and Relationships

Since `NodeModel` and `RelationshipModel` are Pydantic models, you can create
instances, serialize them with `model_dump()`, and deserialize with `model_validate()`.
Pydantic validation applies: missing required fields or wrong types raise errors.

In [8]:
# Create instances
alice = Person(name="Alice", age=32, email="alice@example.com")
inception = Movie(title="Inception", year=2010, rating=8.8)

print("alice.model_dump()     =", alice.model_dump())
print("inception.model_dump() =", inception.model_dump())
print()

# Round-trip through dict
bob_dict = {"name": "Bob", "age": 28}
bob = Person.model_validate(bob_dict)
print("Deserialized Bob:", bob)
print("bob.email is None:", bob.email is None)

alice.model_dump()     = {'name': 'Alice', 'age': 32, 'email': 'alice@example.com'}
inception.model_dump() = {'title': 'Inception', 'year': 2010, 'rating': 8.8}

Deserialized Bob: name='Bob' age=28 email=None
bob.email is None: True


In [9]:
# Pydantic validation errors
from pydantic import ValidationError


try:
    Person(name="Charlie")  # missing required field 'age'
except ValidationError as e:
    print("Missing required field:")
    print(e)

print()

try:
    Person(name="Charlie", age="not_a_number")  # wrong type
except ValidationError as e:
    print("Wrong type:")
    print(e)

Missing required field:
1 validation error for Person
age
  Field required [type=missing, input_value={'name': 'Charlie'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing

Wrong type:
1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not_a_number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing
